# 05 Uncertainty, Calibration & Failure Analysis — Secondary
Brier, ECE, reliability curves, patient-level bootstrap CIs, split conformal. Only run after core result is stable (spec §11).

In [ ]:
import numpy as np
from vist_nrf.utils.seed import set_seed
from vist_nrf.eval.metrics import (evaluate, brier, ece, reliability_curve,
                                       patient_bootstrap, bootstrap_ci,
                                       split_conformal_threshold, conformal_coverage)

set_seed(0)
rng = np.random.default_rng(2)
n = 300
y = (rng.random(n) < 0.2).astype(int)
p = rng.random(n)
pid = np.repeat(np.arange(60), 5)[:n]

r = evaluate(y, p)
print('AUROC=%.3f Brier=%.3f ECE=%.3f prev=%.3f' % (r.auroc, r.brier, r.ece, r.prevalence))
boot = patient_bootstrap(y, p, pid, lambda yt, ys: evaluate(yt, ys).auroc, n_resamples=200, seed=0)
lo, hi = bootstrap_ci(boot)
print('AUROC bootstrap 95%% CI: (%.3f, %.3f)' % (lo, hi))
thr = split_conformal_threshold(p[y == 0], y[y == 0], alpha=0.1)
cov = conformal_coverage(p, y, thr)
print('Conformal threshold=%.3f coverage=%.3f' % (thr, cov))
pt, pp, cnt = reliability_curve(y, p)
print('Reliability bins:', list(zip(pp.round(2), pt.round(2), cnt)))
